In [5]:
import pandas as pd
import re

# Load HZ sheet

file_path = "C:/Users/Ex0164/Book1.xlsx"
hz = pd.read_excel(file_path, sheet_name="HZ")

# Machine columns
machine_cols = [col for col in hz.columns if col.startswith("Machine")]

# Get all unique machines
machines = pd.unique(hz[machine_cols].values.ravel())
machines = [m for m in machines if pd.notna(m)]

# Function to extract tonnage from machine name
def extract_machine_tonnage(machine):
    match = re.search(r'(\d+)T', machine)
    if match:
        return int(match.group(1))
    return None

machine_tonnage = {m: extract_machine_tonnage(m) for m in machines}

# Convert tonnage column into list
def parse_tonnage(x):
    if pd.isna(x):
        return []
    tonnages = []
    for t in str(x).split(","):
        t = t.strip()
        match = re.search(r'(\d+)', t)  # extract digits only
        if match:
            tonnages.append(int(match.group(1)))
    return tonnages


hz["Tonnage_List"] = hz["Tonnage"].apply(parse_tonnage)

# Create compatibility matrix
matrix = []

for _, row in hz.iterrows():
    part = row["Part"]
    tonnage_list = row["Tonnage_List"]

    row_data = {"Part": part}

    for machine in machines:
        m_ton = machine_tonnage[machine]

        if m_ton in tonnage_list:
            row_data[machine] = 1
        else:
            row_data[machine] = 0

    matrix.append(row_data)

compatibility_matrix = pd.DataFrame(matrix)

print(compatibility_matrix)



# Save compatibility matrix to Excel
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"
compatibility_matrix.to_excel(output_path, index=False)

print(f"Compatibility matrix saved to {output_path}")


              Part  BOY-10T-I M027  BOY-10T-II M028  BOY-10T-III M029  \
0    S22127-007A0X               1                1                 1   
1    S22127-007A0X               1                1                 1   
2    S22127-007A0X               1                1                 1   
3    S33081-005A0X               0                0                 0   
4    S33081-005A0X               0                0                 0   
..             ...             ...              ...               ...   
287  S13079-001A1X               0                0                 0   
288  S31841-005A0X               0                0                 0   
289  S33064-004A0X               0                0                 0   
290  S33100-001A1X               0                0                 0   
291  S33100-021A1X               0                0                 0   

     BOY-22T-I M030  BOY-22T-II M031  BOY-22T-III M032  BOY-22T-IV M033  \
0                 1                1            

In [7]:
matrix = []

for _, row in hz.iterrows():
    part = row["Part"]
    tonnage_list = row["Tonnage_List"]

    row_data = {"Part": part}

    # Check if at least one tonnage matches a machine
    matched_tonnages = set()
    for machine in machines:
        m_ton = machine_tonnage[machine]
        if m_ton in tonnage_list:
            matched_tonnages.update(tonnage_list)  # include all tonnages if one matches
            break

    # Assign machines based on matched tonnages
    for machine in machines:
        m_ton = machine_tonnage[machine]
        if m_ton in matched_tonnages:
            row_data[machine] = 1
        else:
            row_data[machine] = 0

    matrix.append(row_data)

compatibility_matrix = pd.DataFrame(matrix)

# Save to Excel
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"
compatibility_matrix.to_excel(output_path, index=False)

print(f"Compatibility matrix saved to {output_path}")


Compatibility matrix saved to C:/Users/Ex0164/compatibility_matrix.xlsx


In [10]:
import pandas as pd
import re

# Load HZ sheet
file_path = "C:/Users/Ex0164/Book1.xlsx"
hz = pd.read_excel(file_path, sheet_name="HZ")

# Machine columns
machine_cols = [col for col in hz.columns if col.startswith("Machine")]

# Get all unique machines
machines = pd.unique(hz[machine_cols].values.ravel())
machines = [m for m in machines if pd.notna(m)]

# Function to extract tonnage from machine name
def extract_machine_tonnage(machine):
    match = re.search(r'(\d+)T', str(machine))
    if match:
        return int(match.group(1))
    return None

machine_tonnage = {m: extract_machine_tonnage(m) for m in machines}

# Convert tonnage column into list
def parse_tonnage(x):
    if pd.isna(x):
        return []
    tonnages = []
    for t in str(x).split(","):
        t = t.strip()
        match = re.search(r'(\d+)', t)  # extract digits only
        if match:
            tonnages.append(int(match.group(1)))
    return tonnages

hz["Tonnage_List"] = hz["Tonnage"].apply(parse_tonnage)

# Create compatibility matrix with extended logic
matrix = []

for _, row in hz.iterrows():
    part = row["Part"]
    tonnage_list = row["Tonnage_List"]

    row_data = {"Part": part}

    # Check if at least one tonnage matches a machine
    matched_tonnages = set()
    for machine in machines:
        m_ton = machine_tonnage[machine]
        if m_ton in tonnage_list:
            matched_tonnages.update(tonnage_list)  # include all tonnages if one matches
            break

    # Assign machines based on matched tonnages
    for machine in machines:
        m_ton = machine_tonnage[machine]
        if m_ton in matched_tonnages:
            row_data[machine] = 1
        else:
            row_data[machine] = 0

    matrix.append(row_data)

compatibility_matrix = pd.DataFrame(matrix)

# Save to Excel
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"
compatibility_matrix.to_excel(output_path, index=False)

print(f"Compatibility matrix saved to {output_path}")


Compatibility matrix saved to C:/Users/Ex0164/compatibility_matrix.xlsx


In [4]:
import pandas as pd

file_path = "C:/Users/Ex0164/Book1.xlsx"
vt = pd.read_excel(file_path, sheet_name="VT")

# Build machine → tonnage mapping directly from VT sheet
machine_tonnage = dict(zip(vt["Machine"], vt["Tonnage"]))

# Get all unique machines
machines = list(machine_tonnage.keys())

matrix = []

for _, row in vt.iterrows():
    part = row["Part"]
    part_ton = row["Tonnage"]   # <-- column where part’s required tonnage is stored

    row_data = {"Part": part}

    for machine in machines:
        if machine_tonnage[machine] == part_ton:
            row_data[machine] = 1
        else:
            row_data[machine] = 0

    matrix.append(row_data)

vt_matrix = pd.DataFrame(matrix)

# Save to Excel
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    vt_matrix.to_excel(writer, sheet_name="VT_Matrix", index=False)

print(f"VT compatibility matrix saved to sheet 'VT_Matrix' in {output_path}")


VT compatibility matrix saved to sheet 'VT_Matrix' in C:/Users/Ex0164/compatibility_matrix.xlsx


In [2]:
import pandas as pd
import re

# Load HZ sheet
file_path = "C:/Users/Ex0164/Book1.xlsx"
hz = pd.read_excel(file_path, sheet_name="HZ")

# Machine columns
machine_cols = [col for col in hz.columns if col.startswith("Machine")]

# Get all unique machines
machines = pd.unique(hz[machine_cols].values.ravel())
machines = [m for m in machines if pd.notna(m)]

# Function to extract tonnage from machine name
def extract_machine_tonnage(machine):
    match = re.search(r'(\d+)T', machine)
    if match:
        return int(match.group(1))
    return None

machine_tonnage = {m: extract_machine_tonnage(m) for m in machines}

# Convert tonnage column into list
def parse_tonnage(x):
    if pd.isna(x):
        return []
    tonnages = []
    for t in str(x).split(","):
        t = t.strip()
        match = re.search(r'(\d+)', t)
        if match:
            tonnages.append(int(match.group(1)))
    return tonnages

hz["Tonnage_List"] = hz["Tonnage"].apply(parse_tonnage)

# ==========================
# BUILD COMPATIBILITY MATRIX
# ==========================

part_machine_map = {}

for _, row in hz.iterrows():
    part = row["Part"]
    tonnage_list = row["Tonnage_List"]

    if part not in part_machine_map:
        part_machine_map[part] = {m:0 for m in machines}

    for machine in machines:
        m_ton = machine_tonnage[machine]
        if m_ton in tonnage_list:
            part_machine_map[part][machine] = 1

# Convert to dataframe
matrix_rows = []
for part, machine_dict in part_machine_map.items():
    row = {"Part":part}
    row.update(machine_dict)
    matrix_rows.append(row)

compatibility_matrix = pd.DataFrame(matrix_rows)

print(compatibility_matrix)

# Save compatibility matrix to Excel
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"
compatibility_matrix.to_excel(output_path, index=False)

print(f"Compatibility matrix saved to {output_path}")


                   Part  BOY-10T-I M027  BOY-10T-II M028  BOY-10T-III M029  \
0         S22127-007A0X               1                1                 1   
1         S33081-005A0X               0                0                 0   
2    14SW410568-00013X0               0                0                 0   
3         S11018-020A0X               0                0                 0   
4         S32047-011A0X               1                1                 1   
..                  ...             ...              ...               ...   
273       S13079-001A1X               0                0                 0   
274       S31841-005A0X               0                0                 0   
275       S33064-004A0X               0                0                 0   
276       S33100-001A1X               0                0                 0   
277       S33100-021A1X               0                0                 0   

     BOY-22T-I M030  BOY-22T-II M031  BOY-22T-III M032  BOY-22T

In [5]:
import pandas as pd

file_path = "C:/Users/Ex0164/Book1.xlsx"
vt = pd.read_excel(file_path, sheet_name="VT")

# Build machine → tonnage mapping (all machines preserved)
machine_tonnage = vt.set_index("Machine")["Tonnage"].to_dict()

# Get all unique machines
machines = vt["Machine"].unique().tolist()

matrix = []

for _, row in vt.iterrows():
    part = row["Part"]
    part_ton = row["Tonnage"]

    row_data = {"Part": part}

    for machine in machines:
        # If machine tonnage matches part tonnage, mark compatible
        row_data[machine] = 1 if machine_tonnage[machine] == part_ton else 0

    matrix.append(row_data)

vt_matrix = pd.DataFrame(matrix)

# Save to Excel
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    vt_matrix.to_excel(writer, sheet_name="VT_Matrix", index=False)

print(f"VT compatibility matrix saved to sheet 'VT_Matrix' in {output_path}")


VT compatibility matrix saved to sheet 'VT_Matrix' in C:/Users/Ex0164/compatibility_matrix.xlsx


In [6]:
import pandas as pd

# Files
vt_file = "C:/Users/Ex0164/Book1.xlsx"
unique_file = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"

# Read VT sheet
vt = pd.read_excel(vt_file, sheet_name="VT")

# Read unique machines
unique = pd.read_excel(unique_file, sheet_name="Sheet1")

# Machine → tonnage from VT
machine_tonnage_vt = vt.set_index("Machine")["Tonnage"].to_dict()

# Machine → tonnage from unique machines file
machine_tonnage_unique = unique.set_index("Unique Machines")["Tonnage"].to_dict()

# Merge both mappings
machine_tonnage = {**machine_tonnage_vt, **machine_tonnage_unique}

# All machines
machines = list(machine_tonnage.keys())

matrix = []

for _, row in vt.iterrows():

    part = row["Part"]
    part_ton = row["Tonnage"]

    row_data = {"Part": part}

    for machine in machines:

        if machine_tonnage[machine] == part_ton:
            row_data[machine] = 1
        else:
            row_data[machine] = 0

    matrix.append(row_data)

vt_matrix = pd.DataFrame(matrix)

# Save
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    vt_matrix.to_excel(writer, sheet_name="VT_Matrix", index=False)

print("Compatibility matrix updated with new machines")

Compatibility matrix updated with new machines


In [5]:
import pandas as pd

# Files
vt_file = "C:/Users/Ex0164/Book1.xlsx"
unique_file = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"

# Read VT sheet
vt = pd.read_excel(vt_file, sheet_name="VT")

# Read unique machines
unique = pd.read_excel(unique_file, sheet_name="Sheet1")

# Machine → tonnage mapping from unique machines file
machine_tonnage = unique.set_index("Unique Machines")["Tonnage"].to_dict()

machines = list(machine_tonnage.keys())
matrix = []

for _, row in vt.iterrows():
    part = row["Part"]
    tons_str = row.get("Tons", None)
    if pd.isna(tons_str):
        continue

    # Split comma-separated tonnages into integers
    part_tons = [int(t.strip()) for t in str(tons_str).split(",") if t.strip().isdigit()]

    row_data = {"Part": part, "Part_Tons": tons_str}
    compatible_list = []

    for machine in machines:
        if machine_tonnage[machine] in part_tons:
            row_data[machine] = 1
            compatible_list.append(machine)
        else:
            row_data[machine] = 0

    row_data["Compatible_Machines_Count"] = len(compatible_list)
    row_data["Compatible_Machines_List"] = ", ".join(compatible_list)

    matrix.append(row_data)

vt_matrix = pd.DataFrame(matrix)

# Save
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    vt_matrix.to_excel(writer, sheet_name="VT_Matrix", index=False)

print("Compatibility matrix updated using comma-separated 'Tons' values")


Compatibility matrix updated using comma-separated 'Tons' values
